In [1]:
import os.path
import gc
import pandas as pd
import pickle as pkl
import torch
import numpy as np
import random
from tqdm import tqdm
import json
from sklearn import metrics
from utils import get_texts_for_ptype
from sklearn import metrics
from scipy.optimize import minimize_scalar
import datasets
from datasets import load_metric

from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import load_dataset

<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.Tensor size changed, may indicate binary incompatibility. Expected 64 from C header, got 80 from PyObject
/home/k2141255/anaconda3/envs/study_design_classifier_bert/lib/python3.11/site-packages/torch/cuda/__init__.py:611: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [5]:
import pandas as pd
results_adjusted_metrics_bert = pd.read_pickle("results_adjusted_metrics_bert.json")

In [14]:
results_adjusted_metrics_bert[-5]

{'ptype': 'is_guideline',
 'model_type': 'distilbert-base-uncased',
 'y_test': 0       1
 1       1
 2       1
 3       1
 4       0
        ..
 7995    0
 7996    1
 7997    1
 7998    0
 7999    1
 Name: is_guideline, Length: 8000, dtype: int64,
 'outputs': PredictionOutput(predictions=array([[-2.979206 ,  3.341626 ],
        [-3.0428305,  3.3597004],
        [ 3.2586763, -3.2003357],
        ...,
        [-2.9148266,  3.245008 ],
        [-2.948446 ,  3.296415 ],
        [ 3.1650624, -3.1516306]], dtype=float32), label_ids=array([1, 1, 0, ..., 1, 1, 0]), metrics={'test_loss': 0.11646104604005814, 'test_accuracy': 0.9695, 'test_precision': 0.9759249873289407, 'test_recall': 0.96275, 'test_f1': 0.9692927258998238, 'test_runtime': 23.2892, 'test_samples_per_second': 343.507, 'test_steps_per_second': 42.938}),
 'db': -1.6180339754913822,
 'auc_roc': 0.50745465625,
 'recall_test': 1.0,
 'spec_test': 0.0,
 'precision_test': 0.005923747122930274,
 'f1_test': 0.011777725975498513}

In [2]:
neg_to_pos_ratio = 1
results_file = "results_titles_abstracts_bert_experiments.json"
    
with open(results_file, "r") as f:
    results_list = json.load(f)
    
inputs = "titles_abstracts"

with open("../data/cleaned_pos_neg_train_val_test_dataset_ids.pkl", "rb") as f:
    prop_typ_id_dict = pkl.load(f)

ptypes = prop_typ_id_dict.keys()
    
metrics_list = []
metric_acc = load_metric("accuracy")
metric_p = load_metric("precision")
metric_r = load_metric("recall")
metric_f = load_metric("f1")

/tmp/ipykernel_211338/2494697924.py:15: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  metric_acc = load_metric("accuracy")
/home/k2141255/anaconda3/envs/study_design_classifier_bert/lib/python3.11/site-packages/datasets/load.py:759: FutureWarning: The repository for accuracy contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.19.1/metrics/accuracy/accuracy.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(


/home/k2141255/anaconda3/envs/study_design_classifier_bert/lib/python3.11/site-packages/datasets/load.py:759: FutureWarning: The repository for precision contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.19.1/metrics/precision/precision.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(


/home/k2141255/anaconda3/envs/study_design_classifier_bert/lib/python3.11/site-packages/datasets/load.py:759: FutureWarning: The repository for recall contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.19.1/metrics/recall/recall.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(


/home/k2141255/anaconda3/envs/study_design_classifier_bert/lib/python3.11/site-packages/datasets/load.py:759: FutureWarning: The repository for f1 contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.19.1/metrics/f1/f1.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(


In [3]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = metric_acc.compute(predictions=predictions, references=labels)['accuracy']
    p = metric_p.compute(predictions=predictions, references=labels)['precision']
    r = metric_r.compute(predictions=predictions, references=labels)['recall']
    f = metric_f.compute(predictions=predictions, references=labels)['f1']
    return {'accuracy': acc, 'precision': p, 'recall': r, 'f1': f}

In [4]:
def get_scores(predictions):
    return [np.exp(p[1])/(np.exp(p[0]) + np.exp(p[1])) for p in predictions]

In [5]:
def predict(scores, db=0):
    return [int(x>=db) for x in scores]

In [6]:
def find_db(db):
    y_pred_test = predict(scores, db)
    recall_test = metrics.recall_score(y_test, y_pred_test, pos_label=1)
    return abs(recall_test - 0.985)

In [7]:
neg_pos_ratio_dict = {
    "is_rct": 5585042/489295,
    "is_sr": 5659292/121063,
    "is_casecontrol": 5628219/65790,
    "is_cohort": 5632460/78688,
    "is_guideline": 5676411/33826
}

def calc_precision(recall, specificity, neg_pos_ratio):
    return recall / (recall + neg_pos_ratio * (1 - specificity) + 1e-15)

def calc_f1(recall, precision):
    return 2*recall*precision/(recall+precision + 1e-15)

In [ ]:
metrics_lst = []
for results_json in results_list:
    ptype = results_json["ptype"]
    model_type = results_json["model_type"]
    def model_init():
        return AutoModelForSequenceClassification.from_pretrained(model_type, return_dict=False, force_download=True)
    X_train, X_val, X_test, y_train, y_val, y_test = get_texts_for_ptype(ptype, inputs="titles_abstracts", neg_to_pos_ratio=neg_to_pos_ratio)
    
    train = pd.DataFrame()
    train['label'] = y_train
    train['text'] = list(X_train)

    val = pd.DataFrame()
    val['label'] = y_val
    val['text'] = list(X_val)

    test = pd.DataFrame()
    test['label'] = y_test
    test['text'] = list(X_test)

    train_dataset = datasets.Dataset.from_pandas(train.reset_index(drop=True))
    val_dataset = datasets.Dataset.from_pandas(val.reset_index(drop=True))
    test_dataset = datasets.Dataset.from_pandas(test.reset_index(drop=True))

    dataset = datasets.dataset_dict.DatasetDict(
        {
            'train': train_dataset,
            'val': val_dataset,
            'test': test_dataset
        }
    )
    
    tokenizer = AutoTokenizer.from_pretrained(model_type)
    def tokenize_function(examples):
        return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=420)

    tokenized_datasets = dataset.map(tokenize_function, batched=True)
    
    train_dataset = tokenized_datasets['train'].map(tokenize_function, batched=True)
    val_dataset = tokenized_datasets['val'].map(tokenize_function, batched=True)
    test_dataset = tokenized_datasets['test'].map(tokenize_function, batched=True)
        
    train_dataset = train_dataset.shuffle(seed=42)
    val_dataset = val_dataset.shuffle(seed=42)
    test_dataset = test_dataset.shuffle(seed=42)
    
    training_args = TrainingArguments(output_dir="test_trainer", 
                                      evaluation_strategy="epoch", 
                                      num_train_epochs=results_json["num_train_epochs"], 
                                      learning_rate=results_json["learning_rate"],
                                      seed=results_json["seed"],
                                      per_device_train_batch_size=results_json["batch_size"],
                                      dataloader_num_workers = 1
                                     )
        
    trainer = Trainer(
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        model_init=model_init
    )

    trainer.train()
    
    outputs = trainer.predict(test_dataset)
    predictions = outputs.predictions
    scores = get_scores(predictions)
    
    auc_roc = metrics.roc_auc_score(y_test, scores)
    
    res = minimize_scalar(find_db)
    db = res["x"]
    y_pred_test = predict(scores, db)
    
    recall_test = metrics.recall_score(y_test, y_pred_test, pos_label=1)
    spec_test = metrics.recall_score(y_test, y_pred_test, pos_label=0)
    precision_test = calc_precision(recall_test, spec_test, neg_pos_ratio_dict[ptype])
    f1_test = calc_f1(recall_test, precision_test)
    
    metrics_dict = {
        "ptype": ptype,
        "model_type": model_type,
        "y_test": y_test,
        "outputs": outputs,
        "db": db,
        "scores": scores,
        "auc_roc": auc_roc,
        "recall_test": recall_test,
        "spec_test": spec_test,
        "precision_test": precision_test,
        "f1_test": f1_test
    }
    
    metrics_lst.append(metrics_dict)
    with open("results_adjusted_metrics_bert.json", "wb") as f:
        pkl.dump(metrics_lst, f)
    
    del trainer
    gc.collect()
    torch.cuda.empty_cache()

19it [00:24,  1.30s/it]/home/k2141255/anaconda3/envs/study_design_classifier_bert/lib/python3.11/site-packages/tqdm/std.py:1182: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
51it [01:05,  1.28s/it]/home/k2141255/anaconda3/envs/study_design_classifier_bert/lib/python3.11/site-packages/tqdm/std.py:1182: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
86it [01:49,  1.27s/it]
57it [01:22,  1.45s/it]


Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

...

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.259400,0.201721,0.929250,0.921453,0.938500,0.929898


Checkpoint destination directory test_trainer/checkpoint-500 already exists and is non-empty.Saving will proceed but saved results may be invalid.
Checkpoint destination directory test_trainer/checkpoint-1000 already exists and is non-empty.Saving will proceed but saved results may be invalid.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Checkpoint destination directory test_trainer/checkpoint-